In [1]:
from config import Config
from DataManager import DataManager
from DataPreprocessor import Preprocessor
from FeatureSelection import FeatureSelector
from ModelManager import ModelManager
from ModelEvaluation import Evaluator

from DataExplorer import Explorer #new


from sklearn.model_selection import train_test_split

import pandas as pd
import numpy as np

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
cfg = Config()
data_manager = DataManager(cfg)
preprocessor = Preprocessor(cfg)

gt_df = data_manager.load_ground_truth()
wl = data_manager.load_wavelengths()

#image = data_manager.load_image(1200)
#processed = preprocessor.process_image(image)

X, y, valid_indices = preprocessor.process_dataset(data_manager, gt_df)
X.shape


(1732, 150)

In [ ]:
image_data = data_manager.load_image(1700)#1700 image - takie samo jak wcześniej
Explorer.compare_masked_vs_not_masked(image_data, band=50)


In [ ]:
Explorer.plot_single_image_spectrum(image_data, wl)

In [ ]:
#Calculating and plotting first, second and third order derivatives of avg spectral reflectance curve
Explorer.plot_image_derivatives(image_data, wl)

In [ ]:
#Calculating correlation between bands of the example image
Explorer.plot_band_correlation_matrix(image_data)

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=cfg.RANDOM_SEED)

In [25]:
selector = FeatureSelector(cfg, n_features=15)
selector.fit(X_train, y_train)

X_train_rfe = selector.transform(X_train)
X_test_rfe = selector.transform(X_test)

Fitting estimator with 150 features.
Fitting estimator with 145 features.
Fitting estimator with 140 features.
Fitting estimator with 135 features.
Fitting estimator with 130 features.
Fitting estimator with 125 features.
Fitting estimator with 120 features.
Fitting estimator with 115 features.
Fitting estimator with 110 features.
Fitting estimator with 105 features.
Fitting estimator with 100 features.
Fitting estimator with 95 features.
Fitting estimator with 90 features.
Fitting estimator with 85 features.
Fitting estimator with 80 features.
Fitting estimator with 75 features.
Fitting estimator with 70 features.
Fitting estimator with 65 features.
Fitting estimator with 60 features.
Fitting estimator with 55 features.
Fitting estimator with 50 features.
Fitting estimator with 45 features.
Fitting estimator with 40 features.
Fitting estimator with 35 features.
Fitting estimator with 30 features.
Fitting estimator with 25 features.
Fitting estimator with 20 features.


In [26]:
models = ModelManager(cfg)

models.tune_fit(X_train_rfe, y_train, verbose=True)
y_pred = models.predict(X_test_rfe)


Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best params for [P] parameter: {'n_estimators': 700, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'max_depth': 50}

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best params for [K] parameter: {'n_estimators': 1000, 'min_samples_split': 2, 'min_samples_leaf': 4, 'max_features': 'log2', 'max_depth': 30}

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best params for [Mg] parameter: {'n_estimators': 1000, 'min_samples_split': 5, 'min_samples_leaf': 4, 'max_features': 'sqrt', 'max_depth': 20}

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best params for [pH] parameter: {'n_estimators': 300, 'min_samples_split': 10, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'max_depth': None}



In [27]:
evaluator = Evaluator(cfg)
y_baseline = ModelManager.get_baseline_predictions(X_train=X_train_rfe, y_train=y_train,X_test=X_test_rfe)
evaluator.calculate_hyperview_score(y_test, y_pred, y_baseline)

P              0.981246
K              0.929610
Mg             0.914848
pH             0.867033
FINAL_SCORE    0.923184
dtype: float64

In [24]:
models_new = ModelManager(cfg)
models_new.load_models('best_rf_model.pkl')

y_pred_new = models_new.predict(X_test_rfe)
evaluator.calculate_hyperview_score(y_test, y_pred_new, y_baseline)

P              0.971106
K              0.946097
Mg             0.905409
pH             0.844258
FINAL_SCORE    0.916717
dtype: float64